# Lab 1: Harness from scratch

[Open in Colab](https://colab.research.google.com/github/dominicdev/test-repo-11/blob/main/labs/lab1_harness.ipynb)

## Lab 1: A harness from scratch (20 min)

An agent is **a model, a loop, tools and state**. By the end you will have a research agent
that decides what to search, which results to read, and when it has enough.

In [ ]:
# Setup: install the workshop package and load your Gemini API key.
import getpass, os, subprocess, sys

REPO = "git+https://github.com/dominicdev/test-repo-11.git@main"
try:
    import agent2027
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", REPO])
    import agent2027

if "GEMINI_API_KEY" not in os.environ:
    try:  # Colab: Secrets panel (key icon) -> add GEMINI_API_KEY
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key (from AI Studio): ")

from agent2027 import config
print("model:", config.MODEL, "| search mode:", config.SEARCH_MODE)

### 1. The tools
Each tool is a name, a description, a schema the model sees, and a Python function.
`web_search` uses DuckDuckGo: the model picks the query, but the **harness** caps how many results
come back and how much page text the model sees.

In [ ]:
from agent2027.tools import core_tools, web_search, fetch_url

for t in core_tools():
    print(f"{t.name:12} {t.description[:70]}")

In [ ]:
# Try the tools by hand first. Snippets help you decide; full pages are what you read.
hits = web_search("Gemini API function calling")
print(hits)

In [ ]:
first_url = hits.split("\n")[1].strip()
print(fetch_url(first_url)[:800])

Note the first line of `fetch_url` output: *untrusted web content*. Search results and fetched pages
are text the agent did not write. Keep that in mind; it is the attack path in the security demo.

### 2. The loop: fill in three gaps
Search for `TODO` below. Each one is one or two lines.
- **TODO 1**: keep the model's turn in the history (`resp.candidates[0].content`)
- **TODO 2**: stop when the model asks for no tools
- **TODO 3**: send each tool result back as a function response

In [ ]:
from google.genai import types
from agent2027.harness import DEFAULT_SYSTEM, user
from agent2027.tools import dispatch

def my_run_agent(task, tools, system_prompt=DEFAULT_SYSTEM, max_steps=15):
    client = config.get_client()
    by_name = {t.name: t for t in tools}
    cfg = types.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[types.Tool(function_declarations=[t.declaration() for t in tools])],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )
    contents = [user(task)]

    for step in range(1, max_steps + 1):
        resp = client.models.generate_content(model=config.MODEL, contents=contents, config=cfg)

        # TODO 1: append the model's turn to `contents`

        calls = resp.function_calls

        # TODO 2: if there are no calls, the model is done: return its text

        parts = []
        for call in calls:
            result = dispatch(by_name, call)  # never raises: errors come back as text
            print(f"[step {step}] {call.name}({dict(call.args)}) -> {len(result)} chars")
            # TODO 3: parts.append(types.Part.from_function_response(...))
        contents.append(types.Content(role="user", parts=parts))

    return "Stopped: max steps reached"

Stuck? Run the next cell to use the finished loop from the package (`agent2027/harness.py`).

In [ ]:
from agent2027.harness import run_agent  # the solution

agent = my_run_agent  # <- switch to `run_agent` if your version is not working yet

### 3. Run it

In [ ]:
TASK = (
    "Research how prompt injection attacks work against LLM agents. Search the web, read at least "
    "three good sources, and write a summary with source URLs to notes.md."
)
answer = agent(TASK, core_tools())
print("\nFINAL:", answer)

In [ ]:
from agent2027.tools import read_file
print(read_file("notes.md"))

### 4. Swap the model: one line
The model is a replaceable part. Change this line and re-run the previous cell. Same harness, different model.

In [ ]:
config.MODEL = "gemini-flash-latest"  # try a smaller or larger model; compare steps, quality, cost

### Things to notice
- **`run_python` is not a sandbox.** It is a subprocess with this machine's permissions. Real isolation needs a sandbox.
- **If search gets throttled**, the harness retries with backoff, then falls back to cached results
  (`config.SEARCH_MODE = "cached"`). Never let one flaky tool stop the loop.
- **Trajectory:** every tool call is logged to `workdir/trajectories/`. Pass `run_id=` to `run_agent` to turn it on.